In [4]:
import numpy as np
import matplotlib
matplotlib.use("Agg")
import matplotlib.pyplot as plt
from matplotlib.backends.backend_pdf import PdfPages

def plot_colour_triangle(
    c_het, c_ref, c_alt,
    labels=("heterozygous", "hom. reference", "hom. alternate"),
    sublabels=("0/1", "0/0", "1/1"),
    out_path="colour_triangle.pdf",
    figsize=(2.8, 2.8),
    resolution=300,
):
    """
    Plot a barycentric colour triangle and save as PDF.

    Parameters
    ----------
    c_het, c_ref, c_alt : str or array-like
        Colours for the three corners. Accepts:
          - hex string:       "#DDAA33"
          - RGB tuple 0-255:  (221, 170, 51)
          - RGB array 0-1:    np.array([0.867, 0.667, 0.2])
    labels : tuple of 3 str
        Corner labels (bold).
    sublabels : tuple of 3 str
        Corner sub-labels (e.g. genotype notation).
    out_path : str
        Output PDF path.
    figsize : tuple
        Figure size in inches.
    resolution : int
        Pixel resolution of the rasterised triangle interior.
    """

    def to_rgb(c):
        if isinstance(c, str) and c.startswith("#"):
            return np.array([int(c[i:i+2], 16) for i in (1, 3, 5)]) / 255
        c = np.asarray(c, dtype=float)
        if c.max() > 1.0:
            c = c / 255
        return c

    ch = to_rgb(c_het)
    cr = to_rgb(c_ref)
    ca = to_rgb(c_alt)

    # Triangle vertices: apex=het (top), left=ref, right=alt
    V = np.array([[0.5, 1.0], [0.0, 0.0], [1.0, 0.0]])

    # Rasterise via barycentric interpolation on a pixel grid
    SZ = resolution
    ys_px, xs_px = np.mgrid[0:SZ, 0:SZ]
    px = xs_px / (SZ - 1)
    py = ys_px / (SZ - 1)

    denom = ((V[1,1]-V[2,1])*(V[0,0]-V[2,0]) + (V[2,0]-V[1,0])*(V[0,1]-V[2,1]))
    w0 = ((V[1,1]-V[2,1])*(px-V[2,0]) + (V[2,0]-V[1,0])*(py-V[2,1])) / denom
    w1 = ((V[2,1]-V[0,1])*(px-V[2,0]) + (V[0,0]-V[2,0])*(py-V[2,1])) / denom
    w2 = 1.0 - w0 - w1
    inside = (w0 >= 0) & (w1 >= 0) & (w2 >= 0)

    img = np.ones((SZ, SZ, 3))
    for i, (ci, wi) in enumerate(zip([ch, cr, ca], [w0, w1, w2])):
        for channel in range(3):
            img[:,:,channel] = np.where(inside,
                img[:,:,channel] - (1.0 - ci[channel]) * wi,
                img[:,:,channel])
    # simpler: build directly
    img = np.ones((SZ, SZ, 3))
    img[:,:,0] = np.where(inside, w0*ch[0] + w1*cr[0] + w2*ca[0], 1.0)
    img[:,:,1] = np.where(inside, w0*ch[1] + w1*cr[1] + w2*ca[1], 1.0)
    img[:,:,2] = np.where(inside, w0*ch[2] + w1*cr[2] + w2*ca[2], 1.0)

    fig, ax = plt.subplots(figsize=figsize)
    fig.patch.set_facecolor("white")
    ax.imshow(img, extent=[0, 1, 0, 1], origin="lower", aspect="equal", zorder=1)

    border = plt.Polygon(V, facecolor="none", edgecolor="#888888", linewidth=0.6, zorder=5)
    ax.add_patch(border)

    LABEL_FS    = 6.5
    SUBLABEL_FS = 5.5
    LABEL_COL   = "#222222"
    SUB_COL     = "#555555"

    corner_layout = [
        (V[0], labels[0], sublabels[0],  0.0,   0.07, "center"),
        (V[1], labels[1], sublabels[1], -0.03, -0.08, "right"),
        (V[2], labels[2], sublabels[2],  0.03, -0.08, "left"),
    ]
    for (vx, vy), lbl, sub, dx, dy, ha in corner_layout:
        ax.text(vx+dx, vy+dy,       lbl, ha=ha, va="center",
                fontsize=LABEL_FS, fontweight="bold", color=LABEL_COL, zorder=10)
        ax.text(vx+dx, vy+dy-0.07, sub, ha=ha, va="center",
                fontsize=SUBLABEL_FS, color=SUB_COL, zorder=10)

    midpoints = [
        ((V[0]+V[1])/2, f"50% {labels[0][:3]} / 50% {labels[1][:3]}", "right", -0.03,  0.0),
        ((V[0]+V[2])/2, f"50% {labels[0][:3]} / 50% {labels[2][:3]}", "left",   0.03,  0.0),
        ((V[1]+V[2])/2, f"50% {labels[1][:3]} / 50% {labels[2][:3]}", "center",  0.0, -0.09),
    ]
    for (mx, my), lbl, ha, dx, dy in midpoints:
        ax.plot(mx, my, "o", ms=2, color="#777777", zorder=6)
        ax.text(mx+dx, my+dy, lbl, ha=ha, va="center",
                fontsize=5, color=SUB_COL, zorder=10)

    ax.set_xlim(-0.20, 1.20)
    ax.set_ylim(-0.20, 1.18)
    ax.set_aspect("equal")
    ax.axis("off")
    plt.tight_layout(pad=0.1)

    with PdfPages(out_path) as pdf:
        pdf.savefig(fig, dpi=300, bbox_inches="tight", facecolor="white")
        d = pdf.infodict()
        d["Title"] = "Barycentric colour triangle"
    plt.close(fig)
    print(f"Saved: {out_path}")


# ── example calls ─────────────────────────────────────────────────────────────

if __name__ == "__main__":

    # Tol palette, lightened 20% (as used in loh_plot_combined.py)
    def lighten(hex_col, amount):
        rgb = np.array([int(hex_col[i:i+2], 16) for i in (1, 3, 5)]) / 255
        return rgb + (1 - rgb) * amount

    plot_colour_triangle(
        c_het = lighten("#DDAA33", 0.2),
        c_ref = lighten("#BB5566", 0.0),
        c_alt = lighten("#004488", 0.0),
        out_path = "loh_color_triangle.pdf",
    )

#     # Hex strings directly
#     plot_colour_triangle(
#         c_het = "#DDAA33",
#         c_ref = "#BB5566",
#         c_alt = "#004488",
#         out_path = "colour_triangle_tol_full.pdf",
#     )

#     # RGB tuples 0-255
#     plot_colour_triangle(
#         c_het = (255, 200, 80),
#         c_ref = (220, 100, 100),
#         c_alt = (80,  130, 200),
#         out_path = "colour_triangle_custom.pdf",
#     )

Saved: loh_color_triangle.pdf
